In [1]:
# Step 2: install kaggle tool + upload your key
!pip install -q kaggle

from google.colab import files
print("When the button appears, upload your kaggle.json:")
files.upload()

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("\n✅ Kaggle key installed.")

When the button appears, upload your kaggle.json:


Saving kaggle.json to kaggle.json

✅ Kaggle key installed.


In [2]:
# Step 3: download + extract the KKBox files we need
import os
os.makedirs("/content/data/raw", exist_ok=True)
os.chdir("/content/data/raw")

COMP = "kkbox-churn-prediction-challenge"

# Download only the corrected v2/v3 files (NOT the 30GB daily log)
!kaggle competitions download -c {COMP} -f train_v2.csv.7z
!kaggle competitions download -c {COMP} -f transactions_v2.csv.7z
!kaggle competitions download -c {COMP} -f user_logs_v2.csv.7z
!kaggle competitions download -c {COMP} -f members_v3.csv.7z

# Unzip them
!apt-get -qq install -y p7zip-full
!7z e train_v2.csv.7z -y
!7z e transactions_v2.csv.7z -y
!7z e user_logs_v2.csv.7z -y
!7z e members_v3.csv.7z -y

os.chdir("/content")
print("\n✅ Files in data/raw:")
!ls -lh /content/data/raw/*.csv

100% 31.3M/31.3M [00:01<00:00, 32.3MB/s]

100% 46.6M/46.6M [00:01<00:00, 38.0MB/s]

100% 654M/654M [00:12<00:00, 53.0MB/s]

100% 231M/231M [00:04<00:00, 49.2MB/s]


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 32818991 bytes (32 MiB)

Extracting archive: train_v2.csv.7z
--
Path = train_v2.csv.7z
Type = 7z
Physical Size = 32818991
Headers Size = 178
Method = LZMA2:24
Solid = -
Blocks = 1

  0%      6% - data/churn_comp_refresh/train_v2.csv                                            15% - data/churn_comp_refresh/train_v2.csv                                            25% - data/churn_co

In [3]:
# Step 4: load labels, check class balance, build the user sample
import pandas as pd, numpy as np

DATA = "/content/data/raw"

train = pd.read_csv(f"{DATA}/train_v2.csv")
print("Shape:", train.shape)
print("Columns:", list(train.columns))
print("Duplicate users:", train["msno"].duplicated().sum())
print("Churn rate:", round(train["is_churn"].mean(), 4))
print()
print(train["is_churn"].value_counts())

Shape: (970960, 2)
Columns: ['msno', 'is_churn']
Duplicate users: 0
Churn rate: 0.0899

is_churn
0    883630
1     87330
Name: count, dtype: int64


In [4]:
# Step 5: stratified sample → freeze the user universe
from sklearn.model_selection import train_test_split

SAMPLE_N = 500_000
sample, _ = train_test_split(
    train,
    train_size=SAMPLE_N,
    stratify=train["is_churn"],   # preserve the 8.99% churn ratio
    random_state=42,              # reproducible
)
sample = sample.reset_index(drop=True)

user_set = set(sample["msno"])    # the filter for every other table

print("Sample shape:", sample.shape)
print("Sample churn rate:", round(sample["is_churn"].mean(), 4), "(should be ~0.0899)")
print("Unique users frozen:", len(user_set))

Sample shape: (500000, 2)
Sample churn rate: 0.0899 (should be ~0.0899)
Unique users frozen: 500000


In [5]:
# Step 6: chunked loader + load members & transactions (filtered)
DATA = "/content/data/raw"

def load_filtered(path, user_set, chunksize=1_000_000):
    """Stream a big CSV in chunks, keep only our sampled users,
    downcast numerics. Peak memory stays bounded regardless of file size."""
    keep = []
    for chunk in pd.read_csv(path, chunksize=chunksize):
        chunk = chunk[chunk["msno"].isin(user_set)]
        for c in chunk.select_dtypes("integer"):
            chunk[c] = pd.to_numeric(chunk[c], downcast="integer")
        for c in chunk.select_dtypes("float"):
            chunk[c] = pd.to_numeric(chunk[c], downcast="float")
        keep.append(chunk)
    return pd.concat(keep, ignore_index=True)

members = load_filtered(f"{DATA}/members_v3.csv", user_set)
txn     = load_filtered(f"{DATA}/transactions_v2.csv", user_set)

print("members:", members.shape)
print("txn:    ", txn.shape)
print("\n=== members sample ===")
print(members.head(3))
print("\n=== transactions sample ===")
print(txn.head(3))

members: (443214, 6)
txn:     (583216, 9)

=== members sample ===
                                           msno  city  bd  gender  \
0  I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=    13  63    male   
1  4De1jAxNRABoyRBDZ82U0yEmzYkqeOugRGVNIf92Xb8=     4  28  female   
2  Z6WIOK9vXy+e2XDBiioNAxuZ0ScXSU/Ebq4tUwqVSrE=    22  38  female   

   registered_via  registration_init_time  
0               9                20110918  
1               9                20110920  
2               9                20110929  

=== transactions sample ===
                                           msno  payment_method_id  \
0  +/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=                 36   
1  +00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=                 41   
2  +0vdYjwKTJKP2OuVUWMqqjFKwp83QG4URcLg1vvFxek=                 41   

   payment_plan_days  plan_list_price  actual_amount_paid  is_auto_renew  \
0                 30              180                 180              1   
1                 3

In [6]:
# Step 7: load user_logs (the 1.4G file) — chunked + filtered
logs = load_filtered(f"{DATA}/user_logs_v2.csv", user_set)

print("logs:", logs.shape)
print("\n=== logs sample ===")
print(logs.head(3))
print("\n=== columns ===")
print(list(logs.columns))

logs: (6970407, 9)

=== logs sample ===
                                           msno      date  num_25  num_50  \
0  nTeWW/eOZA/UHKdD5L7DEqKKFTjaAj3ALLPoAWsU8n0=  20170330       2       2   
1  qR/ndQ5B+1cY+c9ihwLoiz+RFiqEnGyQKo32ZErEVKo=  20170331       3       0   
2  zv12NEuyP2XxcgTDu89mGnPLdnfq8xhBY4jQl8JYLxk=  20170325       5       3   

   num_75  num_985  num_100  num_unq  total_secs  
0       1        0        9       11    2390.699  
1       0        0       39       41    9786.842  
2       2        1       34       43    9060.814  

=== columns ===
['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']


In [7]:
# Step 8: join coverage + does missingness predict churn?
n = len(sample)
users_with_txn  = set(txn["msno"])
users_with_logs = set(logs["msno"])
users_with_mem  = set(members["msno"])

print("Coverage across our 500k sampled users:")
print(f"  have members row:      {len(users_with_mem):>7,} ({len(users_with_mem)/n:.1%})")
print(f"  have transactions:     {len(users_with_txn):>7,} ({len(users_with_txn)/n:.1%})")
print(f"  have listening logs:   {len(users_with_logs):>7,} ({len(users_with_logs)/n:.1%})")

# Is missing-from-logs correlated with churn?
sample = sample.copy()
sample["has_logs"] = sample["msno"].isin(users_with_logs)
sample["has_txn"]  = sample["msno"].isin(users_with_txn)

print("\nChurn rate by log presence:")
print(sample.groupby("has_logs")["is_churn"].mean().round(4))
print("\nChurn rate by transaction presence:")
print(sample.groupby("has_txn")["is_churn"].mean().round(4))

Coverage across our 500k sampled users:
  have members row:      443,214 (88.6%)
  have transactions:     480,853 (96.2%)
  have listening logs:   388,556 (77.7%)

Churn rate by log presence:
has_logs
False    0.0911
True     0.0896
Name: is_churn, dtype: float64

Churn rate by transaction presence:
has_txn
False    0.7872
True     0.0622
Name: is_churn, dtype: float64


In [8]:
# Full date-range inventory across all tables
def span(s, name):
    s = s.dropna()
    if pd.api.types.is_datetime64_any_dtype(s):
        lo, hi = s.min(), s.max()
    else:
        s = s.astype("Int64").dropna()
        lo, hi = int(s.min()), int(s.max())
    print(f"  {name:<28} {lo}  →  {hi}   (n={len(s):,})")

print("=== members ===")
print("  columns:", list(members.columns))
for c in members.columns:
    if "date" in c.lower() or "time" in c.lower() or "init" in c.lower():
        span(members[c], c)

print("\n=== transactions ===")
print("  columns:", list(txn.columns))
for c in ["transaction_date", "membership_expire_date"]:
    span(txn[c], c)

print("\n=== logs ===")
print("  columns:", list(logs.columns))
span(logs["date"], "date")

print("\n=== train (labels) ===")
print("  columns:", list(train.columns), "— no date columns")

=== members ===
  columns: ['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time']
  registration_init_time       20040326  →  20170331   (n=443,214)

=== transactions ===
  columns: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']
  transaction_date             20150101  →  20170331   (n=583,216)
  membership_expire_date       20170227  →  20230517   (n=583,216)

=== logs ===
  columns: ['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']
  date                         20170301  →  20170331   (n=6,970,407)

=== train (labels) ===
  columns: ['msno', 'is_churn'] — no date columns


In [9]:
# Cell A: transaction features (full history)
txn = txn.sort_values(["msno", "transaction_date"])
last = txn.groupby("msno").tail(1).set_index("msno")

tf = pd.DataFrame(index=last.index)
tf["txn_count"]        = txn.groupby("msno").size()
tf["cancel_count"]     = txn.groupby("msno")["is_cancel"].sum()
tf["autorenew_share"]  = txn.groupby("msno")["is_auto_renew"].mean()
tf["total_paid"]       = txn.groupby("msno")["actual_amount_paid"].sum()
tf["last_plan_days"]   = last["payment_plan_days"]
tf["last_actual_paid"] = last["actual_amount_paid"]
tf["last_list_price"]  = last["plan_list_price"]
tf["last_discount"]    = last["plan_list_price"] - last["actual_amount_paid"]
tf["last_auto_renew"]  = last["is_auto_renew"]
tf["last_is_cancel"]   = last["is_cancel"]
tf["payment_method_id"]= last["payment_method_id"]
tf = tf.reset_index()
print("txn features:", tf.shape)
tf.head(3)

txn features: (480853, 12)


,msno,txn_count,cancel_count,autorenew_share,total_paid,last_plan_days,last_actual_paid,last_list_price,last_discount,last_auto_renew,last_is_cancel,payment_method_id
0,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,1,0,1.0,99,30,99,99,0,1,0,41
1,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2,0,1.0,298,30,149,149,0,1,0,39
2,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,1,0,1.0,149,30,149,149,0,1,0,41


In [10]:
# Cell B: listening features (March engagement)
lf = logs.groupby("msno").agg(
    secs_total  = ("total_secs", "sum"),
    secs_mean   = ("total_secs", "mean"),
    unq_mean    = ("num_unq", "mean"),
    active_days = ("date", "nunique"),
    plays_100   = ("num_100", "sum"),
    plays_25    = ("num_25", "sum"),
).reset_index()
lf["completion_ratio"] = lf["plays_100"] / (lf["plays_25"] + 1)
print("log features:", lf.shape)
lf.head(3)

log features: (388556, 8)


,msno,secs_total,secs_mean,unq_mean,active_days,plays_100,plays_25,completion_ratio
0,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,192527.892,6210.577161,28.548387,31,589,191,3.067708
1,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,115411.260,4121.830714,16.714286,28,485,43,11.022727
2,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,149896.558,7137.931333,39.428571,21,436,207,2.096154


In [15]:
# Parse registration_init_time (coerce handles NaN + malformed dates → NaT)
df["registration_init_time"] = pd.to_datetime(
    df["registration_init_time"], format="%Y%m%d", errors="coerce"
)

# Re-run the lines that errored:
df["tenure_days"] = (pd.Timestamp("2017-03-31") - df["registration_init_time"]).dt.days

df["monthly_value"] = df["last_actual_paid"] * (30 / df["last_plan_days"].clip(lower=1))
df["monthly_value"] = df["monthly_value"].replace([np.inf, -np.inf], 0).fillna(0)
df["ltv"] = df["monthly_value"] * 24

print("Final shape:", df.shape)
print("Churn rate:", round(df["is_churn"].mean(), 4))
print("\nChurn by has_txn:\n", df.groupby("has_txn")["is_churn"].mean().round(4))
print("\nChurn by last_auto_renew:\n", df.groupby("last_auto_renew")["is_churn"].mean().round(4))
print("\ntenure_days summary:\n", df["tenure_days"].describe())
print("\nRemaining NaNs:\n", df.isna().sum()[df.isna().sum() > 0])

Final shape: (500000, 31)
Churn rate: 0.0899

Churn by has_txn:
 has_txn
0    0.7872
1    0.0622
Name: is_churn, dtype: float64

Churn by last_auto_renew:
 last_auto_renew
0.0    0.4554
1.0    0.0384
Name: is_churn, dtype: float64

tenure_days summary:
 count    443214.000000
mean       1294.895446
std        1097.849686
min           0.000000
25%         438.000000
50%        1034.000000
75%        1872.000000
max        4753.000000
Name: tenure_days, dtype: float64

Remaining NaNs:
 city                       56786
bd                        301144
age_missing                56786
gender                    299885
registered_via             56786
registration_init_time     56786
payment_method_id          19147
tenure_days                56786
dtype: int64


In [17]:
import json, os
os.makedirs("/content/data/processed", exist_ok=True)

df.to_parquet("/content/data/processed/features_v1.parquet", index=False)

meta = {
    "n_users": int(len(df)),
    "churn_rate": float(df["is_churn"].mean()),
    "n_features": int(df.shape[1] - 2),   # minus msno + is_churn
    "feature_window": "full txn history (2015-2017) + March 2017 logs",
    "known_limitation": "March log features are concurrent with the churn outcome window; leakage-free design using full historical logs is future work",
}
json.dump(meta, open("/content/data/processed/meta_v1.json", "w"), indent=2)
print("Saved:", df.shape)
print(meta)

Saved: (500000, 28)
{'n_users': 500000, 'churn_rate': 0.089942, 'n_features': 26, 'feature_window': 'full txn history (2015-2017) + March 2017 logs', 'known_limitation': 'March log features are concurrent with the churn outcome window; leakage-free design using full historical logs is future work'}


In [19]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [20]:
import os

PROJECT = "/content/drive/MyDrive/churn-prediction-system"

for sub in ["src/churn", "notebooks", "app", "tests", "config",
            "models", "reports/figures", "data/processed", "data/raw"]:
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)

open(f"{PROJECT}/src/churn/__init__.py", "w").close()
print("Project created on Drive:")
!ls -R "{PROJECT}" | head -30

Project created on Drive:
/content/drive/MyDrive/churn-prediction-system:
app
config
data
models
notebooks
reports
src
tests

/content/drive/MyDrive/churn-prediction-system/app:

/content/drive/MyDrive/churn-prediction-system/config:

/content/drive/MyDrive/churn-prediction-system/data:
processed
raw

/content/drive/MyDrive/churn-prediction-system/data/processed:

/content/drive/MyDrive/churn-prediction-system/data/raw:

/content/drive/MyDrive/churn-prediction-system/models:

/content/drive/MyDrive/churn-prediction-system/notebooks:

/content/drive/MyDrive/churn-prediction-system/reports:
figures

/content/drive/MyDrive/churn-prediction-system/reports/figures:


In [21]:
import shutil

# Move the processed feature matrix + metadata to Drive (if they exist)
for f in ["features_v1.parquet", "meta_v1.json"]:
    src = f"/content/data/processed/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{PROJECT}/data/processed/{f}")
        print(f"Copied {f} → Drive")
    else:
        print(f"⚠️ {src} not found — re-run the save cell pointing at PROJECT")

Copied features_v1.parquet → Drive
Copied meta_v1.json → Drive


In [22]:
%%writefile /content/drive/MyDrive/churn-prediction-system/src/churn/features.py
"""Feature engineering for KKBox churn. Imported by notebooks AND the app
so training and serving compute identical features (no train/serve skew)."""
import numpy as np
import pandas as pd

REF_DATE = pd.Timestamp("2017-03-31")


def build_transaction_features(txn: pd.DataFrame) -> pd.DataFrame:
    txn = txn.sort_values(["msno", "transaction_date"])
    last = txn.groupby("msno").tail(1).set_index("msno")
    tf = pd.DataFrame(index=last.index)
    tf["txn_count"]        = txn.groupby("msno").size()
    tf["cancel_count"]     = txn.groupby("msno")["is_cancel"].sum()
    tf["autorenew_share"]  = txn.groupby("msno")["is_auto_renew"].mean()
    tf["total_paid"]       = txn.groupby("msno")["actual_amount_paid"].sum()
    tf["last_plan_days"]   = last["payment_plan_days"]
    tf["last_actual_paid"] = last["actual_amount_paid"]
    tf["last_list_price"]  = last["plan_list_price"]
    tf["last_discount"]    = last["plan_list_price"] - last["actual_amount_paid"]
    tf["last_auto_renew"]  = last["is_auto_renew"]
    tf["last_is_cancel"]   = last["is_cancel"]
    tf["payment_method_id"]= last["payment_method_id"]
    return tf.reset_index()


def build_log_features(logs: pd.DataFrame) -> pd.DataFrame:
    lf = logs.groupby("msno").agg(
        secs_total  = ("total_secs", "sum"),
        secs_mean   = ("total_secs", "mean"),
        unq_mean    = ("num_unq", "mean"),
        active_days = ("date", "nunique"),
        plays_100   = ("num_100", "sum"),
        plays_25    = ("num_25", "sum"),
    ).reset_index()
    lf["completion_ratio"] = lf["plays_100"] / (lf["plays_25"] + 1)
    return lf


def build_feature_matrix(labels, members, txn, logs) -> pd.DataFrame:
    members = members.copy()
    members["bd"] = members["bd"].where(members["bd"].between(10, 90))
    members["age_missing"] = members["bd"].isna().astype("int8")

    tf = build_transaction_features(txn)
    lf = build_log_features(logs)

    df = (labels[["msno", "is_churn"]]
          .merge(members[["msno","city","bd","age_missing","gender",
                          "registered_via","registration_init_time"]], on="msno", how="left")
          .merge(tf, on="msno", how="left")
          .merge(lf, on="msno", how="left"))

    df["has_txn"]  = df["txn_count"].notna().astype("int8")
    df["has_logs"] = df["secs_total"].notna().astype("int8")

    for c in ["txn_count","cancel_count","autorenew_share","total_paid","last_plan_days",
              "last_actual_paid","last_list_price","last_discount","last_auto_renew","last_is_cancel"]:
        df[c] = df[c].fillna(0)
    for c in ["secs_total","secs_mean","unq_mean","active_days","plays_100","plays_25","completion_ratio"]:
        df[c] = df[c].fillna(0)

    df["registration_init_time"] = pd.to_datetime(
        df["registration_init_time"], format="%Y%m%d", errors="coerce")
    df["tenure_days"] = (REF_DATE - df["registration_init_time"]).dt.days

    df["monthly_value"] = df["last_actual_paid"] * (30 / df["last_plan_days"].clip(lower=1))
    df["monthly_value"] = df["monthly_value"].replace([np.inf, -np.inf], 0).fillna(0)
    df["ltv"] = df["monthly_value"] * 24
    return df

Writing /content/drive/MyDrive/churn-prediction-system/src/churn/features.py


In [2]:
# Cell 0: restore session
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np
PROJECT = "/content/drive/MyDrive/churn-prediction-system"

df = pd.read_parquet(f"{PROJECT}/data/processed/features_v1.parquet")
print("Restored:", df.shape)
print("Churn rate:", round(df["is_churn"].mean(), 4))
print("Columns:", list(df.columns))

Mounted at /content/drive
Restored: (500000, 31)
Churn rate: 0.0899
Columns: ['msno', 'is_churn', 'city', 'bd', 'age_missing', 'gender', 'registered_via', 'registration_init_time', 'txn_count', 'cancel_count', 'autorenew_share', 'total_paid', 'last_plan_days', 'last_actual_paid', 'last_list_price', 'last_discount', 'last_auto_renew', 'last_is_cancel', 'payment_method_id', 'secs_total', 'secs_mean', 'unq_mean', 'active_days', 'plays_100', 'plays_25', 'completion_ratio', 'has_txn', 'has_logs', 'tenure_days', 'monthly_value', 'ltv']


In [3]:
# Recompute the 3 derived columns missing from the saved Parquet
df["registration_init_time"] = pd.to_datetime(
    df["registration_init_time"], format="%Y%m%d", errors="coerce"
)
df["tenure_days"] = (pd.Timestamp("2017-03-31") - df["registration_init_time"]).dt.days

df["monthly_value"] = df["last_actual_paid"] * (30 / df["last_plan_days"].clip(lower=1))
df["monthly_value"] = df["monthly_value"].replace([np.inf, -np.inf], 0).fillna(0)
df["ltv"] = df["monthly_value"] * 24

# Re-save so Drive has the COMPLETE matrix this time
df.to_parquet(f"{PROJECT}/data/processed/features_v1.parquet", index=False)
print("Now:", df.shape, "(should be 500000 × 31)")
print("ltv summary:\n", df["ltv"].describe())

Now: (500000, 31) (should be 500000 × 31)
ltv summary:
 count    500000.000000
mean       2978.684192
std         912.921756
min           0.000000
25%        2376.000000
50%        3576.000000
75%        3576.000000
max      107280.000000
Name: ltv, dtype: float64


In [4]:
df = pd.read_parquet(f"{PROJECT}/data/processed/features_v1.parquet")
print("Restored:", df.shape)
print("Churn rate:", round(df["is_churn"].mean(), 4))
print("Columns:", list(df.columns))

Restored: (500000, 31)
Churn rate: 0.0899
Columns: ['msno', 'is_churn', 'city', 'bd', 'age_missing', 'gender', 'registered_via', 'registration_init_time', 'txn_count', 'cancel_count', 'autorenew_share', 'total_paid', 'last_plan_days', 'last_actual_paid', 'last_list_price', 'last_discount', 'last_auto_renew', 'last_is_cancel', 'payment_method_id', 'secs_total', 'secs_mean', 'unq_mean', 'active_days', 'plays_100', 'plays_25', 'completion_ratio', 'has_txn', 'has_logs', 'tenure_days', 'monthly_value', 'ltv']


In [5]:
df.head(5)

,msno,is_churn,city,bd,age_missing,gender,registered_via,registration_init_time,txn_count,cancel_count,...,unq_mean,active_days,plays_100,plays_25,completion_ratio,has_txn,has_logs,tenure_days,monthly_value,ltv
0,y4cg7Cp75/70+cakcZnG8LzGCe6R15LUyhVXxwzzmOU=,0,NaN,NaN,NaN,None,NaN,NaT,1.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,1,0,NaN,99.0,2376.0
1,il0BVINB45kHXPxcUo6iENznCgz7Y9HgIIbImllyB+8=,0,1.0,NaN,1.0,None,7.0,2016-06-24,1.0,0.0,...,58.363636,11.0,748.0,6.0,106.857143,1,1,280.0,99.0,2376.0
2,du3G3Mbku6IqyGpEPVPfdIPNUkG4+GNfF3NwE0eMFis=,0,1.0,NaN,1.0,None,7.0,2012-07-03,1.0,0.0,...,9.250000,20.0,163.0,39.0,4.075000,1,1,1732.0,180.0,4320.0
3,xOMXN07mLhODgABaA9UkZuLtHpwYmPbKwPyn0hUA7+c=,0,NaN,NaN,NaN,None,NaN,NaT,1.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,1,0,NaN,99.0,2376.0
4,BHUbPODBzGqEf2awuWQ2Jg1CiqVXAESPH9Hn62QkYn8=,0,13.0,38.0,0.0,male,7.0,2011-08-05,1.0,0.0,...,15.454545,11.0,190.0,23.0,7.916667,1,1,2065.0,149.0,3576.0


In [6]:
# Cell 1: trajectory transaction features vs churn
print("=== cancel_count vs churn ===")
print(df.groupby(df["cancel_count"].clip(upper=3))["is_churn"].mean().round(4))

print("\n=== autorenew_share (binned) vs churn ===")
df["_ar_bin"] = pd.cut(df["autorenew_share"], [-0.01,0.01,0.5,0.99,1.01],
                       labels=["0%","1-50%","50-99%","100%"])
print(df.groupby("_ar_bin", observed=True)["is_churn"].mean().round(4))

print("\n=== txn_count (binned) vs churn ===")
print(df.groupby(df["txn_count"].clip(upper=5))["is_churn"].mean().round(4))

=== cancel_count vs churn ===
cancel_count
0.0    0.0722
1.0    0.6087
2.0    0.1545
3.0    0.2500
Name: is_churn, dtype: float64

=== autorenew_share (binned) vs churn ===
_ar_bin
0%        0.4560
1-50%     0.4413
50-99%    0.4622
100%      0.0380
Name: is_churn, dtype: float64

=== txn_count (binned) vs churn ===
txn_count
0.0    0.7872
1.0    0.0429
2.0    0.1466
3.0    0.4991
4.0    0.6020
5.0    0.4481
Name: is_churn, dtype: float64


In [7]:
# Cell 2: tenure + engagement vs churn
print("=== tenure (binned, years) vs churn ===")
df["_tenure_yr"] = pd.cut(df["tenure_days"]/365, [0,1,2,4,7,15],
                          labels=["<1y","1-2y","2-4y","4-7y","7y+"])
print(df.groupby("_tenure_yr", observed=True)["is_churn"].mean().round(4))

print("\n=== active_days (March engagement) vs churn ===")
df["_act_bin"] = pd.cut(df["active_days"], [-1,0,5,15,31],
                        labels=["0 days","1-5","6-15","16-31"])
print(df.groupby("_act_bin", observed=True)["is_churn"].mean().round(4))

print("\n=== completion_ratio (binned) vs churn ===")
df["_comp_bin"] = pd.qcut(df["completion_ratio"].clip(upper=df["completion_ratio"].quantile(0.95)),
                          4, labels=["Q1 low","Q2","Q3","Q4 high"], duplicates="drop")
print(df.groupby("_comp_bin", observed=True)["is_churn"].mean().round(4))

=== tenure (binned, years) vs churn ===
_tenure_yr
<1y     0.0835
1-2y    0.0894
2-4y    0.1049
4-7y    0.1010
7y+     0.0905
Name: is_churn, dtype: float64

=== active_days (March engagement) vs churn ===
_act_bin
0 days    0.0911
1-5       0.1259
6-15      0.1010
16-31     0.0755
Name: is_churn, dtype: float64

=== completion_ratio (binned) vs churn ===
_comp_bin
Q1 low     0.0911
Q2         0.0943
Q3         0.0880
Q4 high    0.0864
Name: is_churn, dtype: float64


In [8]:
# Cell 3: correlation ranking + active-user subgroup
num = df.select_dtypes("number").drop(columns=["is_churn"])
corr = num.corrwith(df["is_churn"]).abs().sort_values(ascending=False)
print("=== |correlation| with churn (top 12) ===")
print(corr.head(12).round(3))

active = df[df["has_txn"] == 1]
print(f"\n=== Among users WITH transactions ({len(active):,}, {len(active)/len(df):.0%}) ===")
print("Churn rate in this subgroup:", round(active["is_churn"].mean(), 4))
print("\nTop correlations within active users:")
acn = active.select_dtypes("number").drop(columns=["is_churn"])
print(acn.corrwith(active["is_churn"]).abs().sort_values(ascending=False).head(8).round(3))

=== |correlation| with churn (top 12) ===
has_txn              0.486
autorenew_share      0.480
last_auto_renew      0.480
last_is_cancel       0.379
cancel_count         0.326
ltv                  0.301
monthly_value        0.301
last_plan_days       0.270
last_list_price      0.261
last_actual_paid     0.259
total_paid           0.246
payment_method_id    0.205
dtype: float64

=== Among users WITH transactions (480,853, 96%) ===
Churn rate in this subgroup: 0.0622

Top correlations within active users:
last_is_cancel      0.477
last_plan_days      0.450
last_list_price     0.441
last_actual_paid    0.437
cancel_count        0.416
total_paid          0.388
last_auto_renew     0.316
autorenew_share     0.314
dtype: float64


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [9]:
# Cell 4: define modeling population
# Drop temporary exploration columns
df = df.drop(columns=[c for c in df.columns if c.startswith("_")])

# Scope to active users (transaction history exists) — documented business decision
model_df = df[df["has_txn"] == 1].copy()
print("Active-user population:", model_df.shape)
print("Churn rate:", round(model_df["is_churn"].mean(), 4))

# has_txn is now constant (all 1) → drop it; it's pure within this subgroup
print("has_txn unique values:", model_df["has_txn"].unique())  # should be [1]
model_df = model_df.drop(columns=["has_txn"])

Active-user population: (480853, 31)
Churn rate: 0.0622
has_txn unique values: [1]


In [10]:
# Cell 5: X/y split + column typing
DROP = ["msno", "is_churn", "registration_init_time"]  # id, target, raw date (tenure_days has it)
y = model_df["is_churn"].astype(int)
X = model_df.drop(columns=DROP)

# Identify categorical vs numeric
categorical = ["city", "gender", "registered_via", "payment_method_id"]
categorical = [c for c in categorical if c in X.columns]
numeric = [c for c in X.columns if c not in categorical]

print("Feature count:", X.shape[1])
print("\nCategorical:", categorical)
print("Numeric:", numeric)
print("\nMissing values per feature:\n", X.isna().sum()[X.isna().sum() > 0])

Feature count: 27

Categorical: ['city', 'gender', 'registered_via', 'payment_method_id']
Numeric: ['bd', 'age_missing', 'txn_count', 'cancel_count', 'autorenew_share', 'total_paid', 'last_plan_days', 'last_actual_paid', 'last_list_price', 'last_discount', 'last_auto_renew', 'last_is_cancel', 'secs_total', 'secs_mean', 'unq_mean', 'active_days', 'plays_100', 'plays_25', 'completion_ratio', 'has_logs', 'tenure_days', 'monthly_value', 'ltv']

Missing values per feature:
 city               55876
bd                293730
age_missing        55876
gender            292548
registered_via     55876
tenure_days        55876
dtype: int64


In [11]:
# Cell 5b: remove business-layer artifacts from the feature set
BUSINESS_COLS = ["ltv", "monthly_value"]
X = X.drop(columns=[c for c in BUSINESS_COLS if c in X.columns])

numeric = [c for c in numeric if c not in BUSINESS_COLS]
print("Feature count now:", X.shape[1])
print("Numeric:", len(numeric), "| Categorical:", len(categorical))

Feature count now: 25
Numeric: 21 | Categorical: 4


In [12]:
# Cell 5b: remove business-layer artifacts from the feature set
BUSINESS_COLS = ["ltv", "monthly_value"]
X = X.drop(columns=[c for c in BUSINESS_COLS if c in X.columns])

numeric = [c for c in numeric if c not in BUSINESS_COLS]
print("Feature count now:", X.shape[1])
print("Numeric:", len(numeric), "| Categorical:", len(categorical))

Feature count now: 25
Numeric: 21 | Categorical: 4


In [13]:
# Cell 6: stratified train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Train:", X_train.shape, "churn:", round(y_train.mean(), 4))
print("Test: ", X_test.shape, "churn:", round(y_test.mean(), 4))
print("(both should be ~0.0622)")

Train: (384682, 25) churn: 0.0622
Test:  (96171, 25) churn: 0.0622
(both should be ~0.0622)


In [14]:
# Cell 7: preprocessing ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=20, sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])

# Quick test: fit on train, check output shape
preprocessor.fit(X_train)
print("Preprocessed feature matrix shape:", preprocessor.transform(X_train).shape)
print("(rows unchanged, columns expanded by one-hot encoding)")

Preprocessed feature matrix shape: (384682, 69)
(rows unchanged, columns expanded by one-hot encoding)


In [15]:
# Cell 8: train 4 models, baseline comparison
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# scale_pos_weight for boosters = ratio of negative to positive
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (neg/pos ratio): {spw:.1f}\n")

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=12, n_jobs=-1,
        class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, eval_metric="logloss",
        tree_method="hist", random_state=42),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, random_state=42, verbose=-1),
}

results = []
fitted = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", clf)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    proba = pipe.predict_proba(X_test)[:, 1]
    infer_time = (time.time() - t0) / len(X_test) * 1000  # ms per sample

    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    results.append({"model": name, "ROC_AUC": auc, "PR_AUC": ap,
                    "train_s": train_time, "infer_ms": infer_time})
    fitted[name] = pipe
    print(f"{name:20s} ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}  "
          f"train={train_time:.1f}s  infer={infer_time:.4f}ms")

import pandas as pd
results_df = pd.DataFrame(results).sort_values("PR_AUC", ascending=False)
print("\n", results_df.to_string(index=False))

scale_pos_weight (neg/pos ratio): 15.1

LogisticRegression   ROC-AUC=0.9617  PR-AUC=0.6857  train=26.1s  infer=0.0041ms
RandomForest         ROC-AUC=0.9777  PR-AUC=0.8665  train=109.8s  infer=0.0120ms
XGBoost              ROC-AUC=0.9796  PR-AUC=0.8778  train=21.8s  infer=0.0074ms


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM             ROC-AUC=0.9803  PR-AUC=0.8813  train=15.3s  infer=0.0153ms

              model  ROC_AUC   PR_AUC    train_s  infer_ms
          LightGBM 0.980261 0.881326  15.252851  0.015334
           XGBoost 0.979558 0.877786  21.800486  0.007441
      RandomForest 0.977659 0.866489 109.757199  0.012002
LogisticRegression 0.961736 0.685742  26.112099  0.004090


In [17]:
# Cell 9 (fixed): use SQLite backend instead of deprecated file store
import mlflow, mlflow.sklearn

# SQLite DB on Drive for tracking; artifacts go to a Drive folder
mlflow.set_tracking_uri(f"sqlite:///{PROJECT}/mlflow.db")
mlflow.set_experiment("kkbox-churn-active-users")

for r in results:
    name = r["model"]
    with mlflow.start_run(run_name=f"baseline_{name}"):
        mlflow.set_tag("stage", "baseline")
        mlflow.set_tag("population", "active_users")
        mlflow.log_param("model_type", name)
        mlflow.log_param("imbalance_method", "class_weight/scale_pos_weight")
        mlflow.log_param("n_features_raw", X_train.shape[1])
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_metric("roc_auc", r["ROC_AUC"])
        mlflow.log_metric("pr_auc", r["PR_AUC"])
        mlflow.log_metric("train_seconds", r["train_s"])
        mlflow.log_metric("infer_ms_per_sample", r["infer_ms"])
        mlflow.sklearn.log_model(fitted[name], name="model")

print("✅ Logged 4 baseline runs to MLflow (SQLite backend)")

runs = mlflow.search_runs(experiment_names=["kkbox-churn-active-users"])
print(f"\nTotal runs tracked: {len(runs)}")
print(runs[["tags.mlflow.runName", "metrics.pr_auc", "metrics.roc_auc"]].to_string(index=False))

2026/06/15 17:22:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/15 17:22:28 INFO mlflow.store.db.utils: Updating database tables
2026/06/15 17:22:38 INFO mlflow.tracking.fluent: Experiment with name 'kkbox-churn-active-users' does not exist. Creating a new experiment.
2026/06/15 17:22:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/15 17:22:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative

✅ Logged 4 baseline runs to MLflow (SQLite backend)

Total runs tracked: 4
        tags.mlflow.runName  metrics.pr_auc  metrics.roc_auc
          baseline_LightGBM        0.881326         0.980261
           baseline_XGBoost        0.877786         0.979558
      baseline_RandomForest        0.866489         0.977659
baseline_LogisticRegression        0.685742         0.961736


In [18]:
# Persist Day 8 handoff artifacts
import joblib, os
os.makedirs(f"{PROJECT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT}/models", exist_ok=True)

joblib.dump(
    {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
     "numeric": numeric, "categorical": categorical},
    f"{PROJECT}/data/processed/split_v1.joblib"
)
joblib.dump(fitted["LightGBM"], f"{PROJECT}/models/baseline_lightgbm.joblib")

print("✅ Saved split + best baseline to Drive")
print("processed/:", os.listdir(f"{PROJECT}/data/processed"))
print("models/:", os.listdir(f"{PROJECT}/models"))

✅ Saved split + best baseline to Drive
processed/: ['meta_v1.json', 'features_v1.parquet', 'split_v1.joblib']
models/: ['baseline_lightgbm.joblib']


## Summary — EDA & Baseline Modelling

**Data & scope.** Worked with the KKBox WSDM churn dataset (train_v2 labels, transactions_v2, user_logs_v2, members_v3). Took a stratified 500k-user sample preserving the true 8.99% churn rate. Built a memory-safe chunked loader to handle the large files on Colab.

**Key EDA findings.**
- `has_txn` (whether a user has any transaction record) and auto-renew status are the dominant churn signals. Users without a recent transaction churn at ~79% vs ~6% for those with one; users not on auto-renew churn ~45% vs ~3.8% on auto-renew (a ~12× swing).
- `cancel_count` is **non-monotonic** (one cancellation = high churn, multiple = lower — habitual cancel/resubscribe users), which we predicted would favour tree models over linear ones.
- The March-only `user_logs_v2` data was explored but showed near-flat relationship to churn — a snapshot can't express engagement *trend*. We flagged this for deeper investigation (done in Notebook 03).
- Tenure was **not** protective; engagement features were weak.

**Modelling decisions.**
- **Population:** scoped to *active users* (has_txn==1, ~96% of users, 6.2% churn) — a deliberate, documented choice, since no-transaction accounts are trivially predicted and not actionable for retention.
- **Train/test split:** stratified **random** 80/20 (seed 42), *not* temporal. Temporal leakage control was handled in feature engineering (pre-prediction history only), not via a dated split.
- **Pipeline:** sklearn `ColumnTransformer` (median-impute + scale numerics; mode-impute + one-hot categoricals) bundled with the model in one object, so preprocessing travels with the model to serving — no train/serve skew.

**Why LightGBM.** Compared four models on PR-AUC (the right metric under 6.2% imbalance — ROC-AUC is too generous here):

| Model | PR-AUC | Train time |
|---|---|---|
| **LightGBM** | **0.881** | **15s** |
| XGBoost | 0.878 | 22s |
| RandomForest | 0.866 | 110s |
| LogisticRegression | 0.686 | 26s |

The three tree models clustered at the top (confirming the EDA prediction that non-monotonic features beat linear models by ~0.19 PR-AUC). **LightGBM won on both accuracy and speed** — matching/beating RandomForest's accuracy at ~7× faster training — so it became our model of choice. All four runs logged to MLflow.